
# 42 国内商品期货期权：合约、保证金与期权链

> **本章要解决的问题**：一张商品期权合约到底规定了什么？卖方一手要压多少保证金？同一张链上，为什么“看着便宜”的期权可能更贵？行权之后，手里拿到的是什么？

## 本章导读

第24章讲期权怎么定价、希腊字母怎么算；这一章往前走一步，看**国内商品期货期权在交易所里长什么样、一手账怎么记**。

内容顺序：交易所版图 → 合约规则（行权价间距、最小变动、最后交易日、行权方式）→ 保证金公式与手算 → 一手账本（支付/盈亏）→ 期权链与流动性 → 行权/指派与商品季节性。

**重要说明**：交易所参数会调整，本课数值都是**教学示例参数**，只用来练算法，不能当交易所公告；真实下单前必须查合约细则与期货公司通知。

### 学习目标

- **任务 1**：拿到一张商品期权合约，逐项读出标的期货、类型、行权价、报价单位、合约乘数、最小变动价位、行权方式、最后交易日。
- **任务 2**：用教学示例参数手算卖方一手保证金，并与买方权利金对比资金占用。
- **任务 3**：把“每吨权利金”换算成“一手权利金”，算出到期支付、盈亏平衡与净盈亏。
- **任务 4**：从一条期权链里筛出平值附近、价差小、有成交的行权价，并读隐含波动率偏斜。
- **任务 5**：判断到期前该行权还是平仓，算出行权后期货头寸的保证金。
- **任务 6**：做一次季节性检查，并说明它为什么会过拟合。

### 先修

第24章的 Black–76、平价、希腊字母；第09章的期限结构与换月；会 pandas/numpy。

---

## 在量化研究中的位置

- **谁用**：期权研究员、做市/交易员、产业客户（套保）、风控、结算。
- **输入**：交易所合约细则、期货/期权行情、持仓量/成交量、保证金率、手续费。
- **输出**：合约规格表、一手账本、保证金占用、链筛选结果、行权/指派决策。
- **在流程的哪一步**：第24章定价之后、第27章对冲之前；从“模型价”落到“交易所口径”的一步。
- **失败模式**：教学参数当真实参数、乘数与单位错、最后交易日记错、虚值额算反、忽略行权后保证金、季节性当因果。
- **最小工作流**：读细则 → 规格表 → 保证金 → 一手账本 → 链筛选 → 行权/到期管理。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="42 国内商品期货期权：合约、保证金与期权链" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>42 国内商品期货期权：合约、保证金与期权链｜知识图</title>
<desc>本章主线：先读合约规格，再按公式算一手保证金，最后在期权链里挑行权价与流动性。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a42" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao42" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag42" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">42</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">国内商品期货期权：合约、保证金与期权链</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：先读合约规格，再按公式算一手保证金，最后在期权链里挑行权价与流动性。</text>
<rect x="40" y="88" width="240" height="290" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="54" y="108" font-size="10.5" fill="#1B3A5C" font-weight="bold">① 合约规格（示例）</text>
<text x="54" y="130" font-size="8.6" fill="#6E7B8B">合约乘数</text>
<text x="54" y="147" font-size="10" fill="#1B3A5C" font-weight="bold">10 吨/手</text>
<line x1="54" y1="156" x2="266" y2="156" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="54" y="170" font-size="8.6" fill="#6E7B8B">最小变动</text>
<text x="54" y="187" font-size="10" fill="#1B3A5C" font-weight="bold">0.5 元/吨</text>
<line x1="54" y1="196" x2="266" y2="196" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="54" y="210" font-size="8.6" fill="#6E7B8B">行权方式</text>
<text x="54" y="227" font-size="10" fill="#1B3A5C" font-weight="bold">美式</text>
<line x1="54" y1="236" x2="266" y2="236" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="54" y="250" font-size="8.6" fill="#6E7B8B">最后交易日</text>
<text x="54" y="267" font-size="10" fill="#1B3A5C" font-weight="bold">标的期货前第 5 个交易日</text>
<line x1="54" y1="276" x2="266" y2="276" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="54" y="290" font-size="8.6" fill="#6E7B8B">行权价间距</text>
<text x="54" y="307" font-size="10" fill="#1B3A5C" font-weight="bold">依交易所规则</text>
<line x1="54" y1="316" x2="266" y2="316" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<text x="54" y="330" font-size="8.6" fill="#6E7B8B">保证金</text>
<text x="54" y="347" font-size="10" fill="#1B3A5C" font-weight="bold">按公式收取</text>
<line x1="54" y1="356" x2="266" y2="356" stroke="#E6EFF7" stroke-width="0.9" stroke-linecap="round"/>
<rect x="295" y="88" width="250" height="290" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="309" y="108" font-size="10.5" fill="#1B3A5C" font-weight="bold">② 保证金 / 手</text>
<text x="309" y="134" font-size="9.2" fill="#1B3A5C">权利金 = 期权价 × 乘数</text>
<text x="309" y="153" font-size="9.2" fill="#1B3A5C">保证金 = 权利金 +</text>
<text x="309" y="172" font-size="9.2" fill="#6E7B8B">  max(标的保证金 − 虚值额, 最低)</text>
<rect x="309" y="200" width="200" height="1" fill="#E6EFF7"/>
<text x="309" y="222" font-size="8.6" fill="#6E7B8B">期权价</text>
<text x="470" y="222" font-size="9.2" fill="#1B3A5C" text-anchor="end">20 元/吨</text>
<text x="309" y="244" font-size="8.6" fill="#6E7B8B">权利金</text>
<text x="470" y="244" font-size="9.2" fill="#1B3A5C" text-anchor="end">200 元</text>
<text x="309" y="266" font-size="8.6" fill="#6E7B8B">标的保证金</text>
<text x="470" y="266" font-size="9.2" fill="#1B3A5C" text-anchor="end">3,150 元</text>
<text x="309" y="288" font-size="8.6" fill="#6E7B8B">虚值额</text>
<text x="470" y="288" font-size="9.2" fill="#1B3A5C" text-anchor="end">150 元</text>
<text x="309" y="310" font-size="8.6" fill="#6E7B8B">保证金</text>
<text x="470" y="310" font-size="9.2" fill="#E07B39" text-anchor="end" font-weight="bold">3,200 元/手</text>
<rect x="309" y="336" width="200" height="16" fill="#E6EFF7" rx="4"/>
<rect x="309" y="336" width="12" height="16" fill="#E07B39" rx="4"/>
<text x="309" y="364" font-size="8.4" fill="#6E7B8B">权利金只占保证金的一部分</text>
<rect x="565" y="88" width="195" height="290" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="578" y="108" font-size="10.5" fill="#1B3A5C" font-weight="bold">③ 期权链</text>
<rect x="578" y="124" width="182" height="22" fill="#1B3A5C" rx="4"/>
<text x="597" y="139" font-size="8" fill="#FFFFFF" text-anchor="middle">K</text>
<text x="659" y="139" font-size="8" fill="#FFFFFF" text-anchor="middle">Call 买/卖</text>
<text x="731" y="139" font-size="8" fill="#FFFFFF" text-anchor="middle">IV</text>
<text x="597" y="163" font-size="8.6" fill="#1B3A5C" text-anchor="middle">90</text>
<text x="659" y="163" font-size="8.6" fill="#1B3A5C" text-anchor="middle">12.5 / 13.5</text>
<text x="731" y="163" font-size="8.6" fill="#1B3A5C" text-anchor="middle">31%</text>
<text x="597" y="189" font-size="8.6" fill="#1B3A5C" text-anchor="middle">95</text>
<text x="659" y="189" font-size="8.6" fill="#1B3A5C" text-anchor="middle">9.2 / 10.0</text>
<text x="731" y="189" font-size="8.6" fill="#1B3A5C" text-anchor="middle">29%</text>
<rect x="578" y="198" width="182" height="26" fill="#E6EFF7" rx="3"/>
<text x="597" y="215" font-size="8.6" fill="#1B3A5C" text-anchor="middle">100</text>
<text x="659" y="215" font-size="8.6" fill="#1B3A5C" text-anchor="middle">6.4 / 7.0</text>
<text x="731" y="215" font-size="8.6" fill="#1B3A5C" text-anchor="middle">27%</text>
<text x="597" y="241" font-size="8.6" fill="#1B3A5C" text-anchor="middle">105</text>
<text x="659" y="241" font-size="8.6" fill="#1B3A5C" text-anchor="middle">4.2 / 4.8</text>
<text x="731" y="241" font-size="8.6" fill="#1B3A5C" text-anchor="middle">28%</text>
<text x="578" y="300" font-size="8.2" fill="#6E7B8B">Call 卖价高 → 买价隐波低</text>
<text x="578" y="316" font-size="8.2" fill="#1B3A5C">选行权价：近月 ATM 流动性最好</text>
<text x="578" y="332" font-size="8.2" fill="#6E7B8B">买卖价差 = 隐形的交易成本</text>
<path d="M 281 230 L 294 230" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag42)"/>
<path d="M 546 230 L 564 230" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag42)"/>
</svg>

> **图**：本章主线：先读合约规格，再按公式算一手保证金，最后在期权链里挑行权价与流动性。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 四家交易所与品种版图

### 1.1 谁在交易什么

| 交易所 | 简称 | 商品期权品种（举例） | 备注 |
|---|---|---|---|
| 上海期货交易所 | SHFE | 铜、铝、锌、黄金、白银、螺纹钢、天然橡胶等 | 标的为相应期货合约 |
| 大连商品交易所 | DCE | 豆粕、玉米、铁矿石、棕榈油、LPG 等 | 农产品与工业品 |
| 郑州商品交易所 | CZCE | 白糖、棉花、PTA、甲醇、菜籽粕等 | 农产品与化工 |
| 上海国际能源交易中心 | INE | 原油、20 号胶、低硫燃料油、国际铜等 | 国际化品种 |
| 广州期货交易所 | GFEX | 工业硅、碳酸锂等 | 新品种，期权陆续上市 |

品种清单以交易所官网为准；做研究的第一步不是定价，而是确认标的、合约月份、行权方式、最后交易日。

### 1.2 商品期权和股票期权的三个不同

1. **标的是期货合约**：有到期日、有保证金、要换月、可能交割；
2. **行权得到的是期货头寸**，不是现金；行权之后马上要交期货保证金；
3. **卖方保证金按“期货保证金 + 权利金 − 虚值额”算**，不是固定的百分比。

这三条决定了商品期权的账比股票期权复杂：同一笔观点，保证金、换月、行权规则都会改变实际盈亏。

---

## 2. 合约规则：六个必须读的字段

### 2.1 行权价间距

行权价按间距挂出：标的价格越低、间距通常越小。间距太宽，找不到精确执行价；间距太窄，链太长、流动性分散。标的涨跌后交易所会增挂新行权价，所以链是动态的。

### 2.2 最小变动价位与一跳价值

$$\text{一跳价值}=\text{最小变动价位}\times\text{合约乘数}.$$

**公式拆解**
- **符号**：最小变动价位是报价能变动的最小单位（元/吨、元/克等），合约乘数是每手对应的标的数量（吨/手等），一跳价值是报价跳一格时一手的现金变化（元/手）。
- **结构**：一跳价值等于最小变动价位乘以合约乘数，同理一手名义价值等于报价乘以合约乘数，乘数是一手账本的开关。
- **代入**：螺纹钢最小变动价位 1 元/吨乘 10 吨/手得 10 元，黄金 0.02 元/克乘 1000 克/手得 20 元，原油 0.1 元/桶乘 1000 桶/手得 100 元。
- **边界**：最小变动价位不等于行权价间距，深虚值期权可能长期停在一个价位，一跳价值只是成本下限、实际滑点常是几跳。
- **用途**：算最小盈亏单位、合约规模与跨品种比较；不能把每吨报价直接当一手价格，漏乘乘数是新手最常见的账本错误。

报价单位可能是元/吨、元/克、元/桶、元/500 千克。**一手账本的第一步永远是乘乘数**：每吨报价乘以吨数，才是现金。

### 2.3 最后交易日与到期日

商品期权的最后交易日通常**早于**标的期货交割月，给行权后的期货头寸留出处理时间。到期日与最后交易日可能相同也可能不同，行权与放弃另有截止时间。临近到期流动性下降、保证金可能上调。

### 2.4 行权方式

- **美式**：到期前任一交易日都可以行权；
- **欧式**：只能在到期日行权。

国内商品期权两种都有，**同一交易所不同品种也可能不同**。不要凭记忆套用，逐合约查细则；行权方式直接影响提前行权与被指派的风险。

### 2.5 代码：合约规格表



In [2]:

import pandas as pd

spec = pd.DataFrame([
    ["SHFE", "螺纹钢", "rb2701", "元/吨", 10, 1.0, 50],
    ["SHFE", "黄金", "au2702", "元/克", 1000, 0.02, 4],
    ["DCE", "豆粕", "m2701", "元/吨", 10, 0.5, 50],
    ["CZCE", "白糖", "SR701", "元/吨", 10, 0.5, 100],
    ["INE", "原油", "sc2701", "元/桶", 1000, 0.1, 5],
], columns=["交易所", "品种", "标的期货", "报价单位", "合约乘数", "最小变动", "行权价间距"])

price = pd.Series([3500.0, 620.0, 3200.0, 5900.0, 520.0], index=spec["品种"])
spec["期货价"] = price.values
spec["一手名义"] = spec["期货价"] * spec["合约乘数"]
spec["一跳价值"] = spec["最小变动"] * spec["合约乘数"]

print("【教学示例参数，非交易所公告；真实参数以交易所合约细则为准】")
print(spec.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))


【教学示例参数，非交易所公告；真实参数以交易所合约细则为准】
 交易所  品种   标的期货 报价单位  合约乘数  最小变动  行权价间距      期货价       一手名义   一跳价值
SHFE 螺纹钢 rb2701  元/吨    10  1.00     50 3,500.00  35,000.00  10.00
SHFE  黄金 au2702  元/克  1000  0.02      4   620.00 620,000.00  20.00
 DCE  豆粕  m2701  元/吨    10  0.50     50 3,200.00  32,000.00   5.00
CZCE  白糖  SR701  元/吨    10  0.50    100 5,900.00  59,000.00   5.00
 INE  原油 sc2701  元/桶  1000  0.10      5   520.00 520,000.00 100.00



**读数与应用**：同样是“一手”，螺纹钢名义价值 3.5 万元、黄金 62 万元、原油 52 万元，差一个数量级；一跳价值从 5 元到 100 元不等。名义价值决定保证金和资金占用，一跳价值决定最小报价单位和交易成本——**两者都看，才算读完一张合约。**

**常见错误**：

1. 把每吨/每克报价直接当一手价格，漏乘合约乘数。
2. 把“行权价间距”当“最小变动价位”，一个是链的密度、一个是报价精度。
3. 用近月期权的参数配远月期货的行情。

---

## 3. 保证金：卖方一手要压多少钱

### 3.1 买方与卖方的资金占用

| | 买方 | 卖方 |
|---|---|---|
| 开仓现金流 | 付权利金 | 收权利金 |
| 保证金 | 无 | 有，随标的价格、波动率变化 |
| 最大亏损 | 权利金 | 可能很大 |
| 每日风险 | 权利金已付清 | 每日盯市、可能追加 |

### 3.2 公式（教学示例口径）

$$\text{权利金市值}=P_{\text{结算}}\times M,$$

$$\text{标的期货保证金}=F\times M\times r,$$

$$\text{虚值额}=\begin{cases}\max(K-F,0)\times M, & \text{看涨}\\[2pt] \max(F-K,0)\times M, & \text{看跌}\end{cases}$$

$$\text{卖方保证金}=\max\big(\text{权利金市值}+\text{标的期货保证金}-0.5\times\text{虚值额},\ \text{权利金市值}+0.5\times\text{标的期货保证金}\big).$$

**公式拆解**
- **符号**：权利金市值是结算价乘乘数，标的期货保证金是 $F\times M\times r$，虚值额按看涨或看跌取 max 再乘乘数，$K$ 是行权价。
- **结构**：取“权利金市值加期货保证金减半个虚值额”与“权利金市值加半个期货保证金”的较大者，前者做实值程度调整、后者是下限。
- **代入**：$F=3500$、$M=10$、$r=10\%$、$K=3600$ 看涨、权利金 55 元/吨时，权利金市值 550、期货保证金 3500、虚值额 1000，保证金为 3550 元。
- **边界**：深度实值时虚值额为 0、保证金约等于权利金加期货保证金，交易所系数与组合保证金优惠另算。
- **用途**：估算卖方策略的资金占用与追保风险；这不是交易所精确规则，保证金每天按结算价重算。

这是国内商品期权常见的“max + 虚值额扣减”结构，但各交易所的具体系数、用结算价还是收盘价、组合保证金优惠都不同。**本课用教学示例系数，真实计算以交易所与期货公司为准。**

### 3.3 手算一遍

螺纹钢教学示例：$F=3500$ 元/吨，$M=10$ 吨/手，期货保证金率 $r=10\%$，卖出一手行权价 3600 的看涨，权利金结算价 55 元/吨。

- 权利金市值：$55\times10=550$ 元；
- 标的期货保证金：$3500\times10\times10\%=3500$ 元；
- 虚值额：$\max(3600-3500,0)\times10=1000$ 元；
- 保证金：$\max(550+3500-500,\ 550+1750)=\max(3550,2300)=3550$ 元。

买方只付 550 元权利金，卖方要压 3550 元——**权利金远小于保证金占用**，这是卖方策略“收益有限、风险无限”的资金侧证据。

### 3.4 代码：保证金手算与敏感性



In [3]:

def seller_margin(F, K, prem, mult, fut_rate=0.10, kind="call"):
    premium_value = prem * mult
    fut_margin = F * mult * fut_rate
    otm = (max(K - F, 0) if kind == "call" else max(F - K, 0)) * mult
    margin = max(premium_value + fut_margin - 0.5 * otm,
                 premium_value + 0.5 * fut_margin)
    return margin, otm


F, mult, fut_rate = 3500.0, 10, 0.10
print(f"教学示例：标的 {F:.0f} 元/吨，乘数 {mult}，期货保证金率 {fut_rate:.0%}")
print(f"{'行权价':>6}{'类型':>4}{'权利金':>8}{'虚值额':>8}{'保证金':>10}{'买方权利金':>10}")
for K, prem, kind in [(3400, 150, "call"), (3500, 90, "call"), (3600, 55, "call"),
                      (3400, 45, "put"), (3500, 90, "put"), (3600, 160, "put")]:
    margin, otm = seller_margin(F, K, prem, mult, fut_rate, kind)
    label = "看涨" if kind == "call" else "看跌"
    print(f"{K:>6}{label:>4}{prem:>8.0f}{otm:>8.0f}{margin:>10.0f}{prem * mult:>10.0f}")


教学示例：标的 3500 元/吨，乘数 10，期货保证金率 10%
   行权价  类型     权利金     虚值额       保证金     买方权利金
  3400  看涨     150       0      5000      1500
  3500  看涨      90       0      4400       900
  3600  看涨      55    1000      3550       550
  3400  看跌      45    1000      3450       450
  3500  看跌      90       0      4400       900
  3600  看跌     160       0      5100      1600



**读数与应用**：看涨期权越虚值，虚值额越大、保证金越低；越实值，保证金越接近“期货保证金 + 权利金”。同一行权价，看跌与看涨的保证金也不同，因为虚值额的方向相反。保证金随行情每天重算，**卖方的资金占用是动态的，不是开仓时那一个数。**

**常见错误**：

1. 把虚值额算反：看涨的虚值额是 $\max(K-F,0)$，看跌是 $\max(F-K,0)$。
2. 忘掉标的期货保证金那一项，只算权利金。
3. 用收盘价算出保证金，就以为盘中不会被追保。
4. 忽略组合保证金优惠：单腿相加会高估保证金，真实占用要看交易所规则。

**停下来想一想**：深度实值的看涨期权，卖方保证金更接近“权利金 + 期货保证金”，还是更接近“权利金 + 半个期货保证金”？

<details><summary>对照你的思路</summary>

接近“权利金 + 期货保证金”。深度实值时虚值额为 0，减项不起作用；深度虚值时虚值额大，减项把保证金压到第二项附近。所以保证金随实值程度变化，不是固定比例。

</details>

---

## 4. 支付、盈亏与一手账本

### 4.1 到期支付与盈亏平衡

到期标的期货价为 $F_T$：

$$\text{看涨支付}=\max(F_T-K,0)\times M,\qquad \text{看跌支付}=\max(K-F_T,0)\times M.$$

**公式拆解**
- **符号**：$F_T$ 是到期时标的期货价格，$K$ 是行权价，$M$ 是合约乘数，支付单位是元/手，时间顺序是先到期再按内在价值结算。
- **结构**：看涨支付为 $\max(F_T-K,0)\times M$、看跌为 $\max(K-F_T,0)\times M$，支付是毛额、买方盈亏还要减去已付权利金与手续费。
- **代入**：$K=3600$、$M=10$、权利金 55 元/吨，若到期 $F_T=3700$ 则支付 1000 元、净赚 450 元，盈亏平衡点是 $3600+55=3655$ 元/吨。
- **边界**：$F_T=K$ 时支付为 0、买方最大亏损就是权利金，卖方看涨亏损无上限，行权后拿到期货头寸仍要交保证金。
- **用途**：算到期盈亏与盈亏平衡点、评估卖方风险；不能只看支付不看权利金，未到期期权还有时间价值。

买方盈亏 = 支付 − 权利金×$M$ − 手续费；卖方相反。盈亏平衡点：看涨 $K+$每吨权利金，看跌 $K-$每吨权利金。**注意这是“每吨”的平衡点，换算成一手要再乘乘数。**

### 4.2 一手账本模板

| 项目 | 公式 | 说明 |
|---|---|---|
| 一手权利金 | 每吨报价 × 乘数 | 买方付出、卖方收入 |
| 到期支付 | 内在价值 × 乘数 | 到期才结算 |
| 买方净盈亏 | 支付 − 一手权利金 − 手续费 | 最大亏损是权利金 |
| 卖方净盈亏 | 一手权利金 − 支付 − 手续费 | 亏损可能很大 |
| 行权后期货保证金 | $F_T\times M\times r$ | 行权不是结束 |

### 4.3 代码：支付、盈亏与一手账本



In [4]:

import numpy as np
from scipy.special import ndtr


def black76(F, K, T, vol, r=0.02, kind="call"):
    if T <= 0:
        return max(F - K, 0) if kind == "call" else max(K - F, 0)
    w = vol * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * w * w) / w
    d2 = d1 - w
    if kind == "call":
        return float(np.exp(-r * T) * (F * ndtr(d1) - K * ndtr(d2)))
    return float(np.exp(-r * T) * (K * ndtr(-d2) - F * ndtr(-d1)))


F, K, T, vol, mult = 3500.0, 3500.0, 30 / 365, 0.20, 10
prem = black76(F, K, T, vol)
print(f"教学权利金 {prem:.2f} 元/吨，一手 {prem * mult:.2f} 元")
print(f"看涨盈亏平衡 {K + prem:.2f} 元/吨；看跌盈亏平衡 {K - prem:.2f} 元/吨")
print(f"{'到期价':>6}{'看涨买方':>10}{'看涨卖方':>10}{'看跌买方':>10}{'看跌卖方':>10}")
for FT in [3200, 3400, 3500, 3600, 3800]:
    call_pnl = (max(FT - K, 0) - prem) * mult
    put_pnl = (max(K - FT, 0) - prem) * mult
    print(f"{FT:>6}{call_pnl:>10.0f}{-call_pnl:>10.0f}{put_pnl:>10.0f}{-put_pnl:>10.0f}")


教学权利金 79.92 元/吨，一手 799.19 元
看涨盈亏平衡 3579.92 元/吨；看跌盈亏平衡 3420.08 元/吨
   到期价      看涨买方      看涨卖方      看跌买方      看跌卖方
  3200      -799       799      2201     -2201
  3400      -799       799       201      -201
  3500      -799       799      -799       799
  3600       201      -201      -799       799
  3800      2201     -2201      -799       799



**读数与应用**：平值权利金约 80 元/吨、一手 800 元；到期涨到 3800，看涨买方净赚 $(300-80)\times10=2200$ 元，卖方相反。买方最大亏损是一手权利金，卖方在标的大涨时亏损没有上限。一手账本的三个易错点：乘数、每吨与一手、手续费与行权后的保证金。

**常见错误**：

1. 把每吨权利金当一手权利金（漏乘乘数）。
2. 把“支付”当“净利”，忘了扣权利金。
3. 用平值价当盈亏平衡点，忽略权利金。

---

## 5. 期权链：从 T 型报价到可交易的行权价

### 5.1 T 型报价

期权链按 T 型排列：中间行权价，左看涨、右看跌，每个方向有买价、卖价、最新价、持仓量、成交量。**链不是价格表，而是流动性地图。**

### 5.2 链上要看的五个量

1. **买卖价差占中间价比例**：比绝对价差更能反映成本；
2. **持仓量与成交量**：有没有人跟你交易；
3. **近月/远月**：时间价值与流动性此消彼长；
4. **平值/虚值**：虚值权利金小，价差占比往往更大；
5. **隐含波动率**：把不同行权价拉到同一尺度上比较贵贱。

### 5.3 筛选规则

- 只保留平值上下 $N$ 档；
- 价差占比小于阈值（例如 6%，教学示例）；
- 持仓量与成交量高于下限；
- 对剩余行权价反解 IV，看微笑/偏斜。

### 5.4 代码：造一条教学链并筛选



In [5]:

import numpy as np
import pandas as pd
from scipy.optimize import brentq

F, T, r = 3500.0, 30 / 365, 0.02
strikes = np.arange(3200, 3801, 50)
rows = []
for K in strikes:
    x = K / F - 1
    vol = 0.20 - 0.45 * x + 1.5 * x * x        # 教学偏斜：低行权价 IV 更高
    for kind in ["C", "P"]:
        mid = black76(F, K, T, vol, r, "call" if kind == "C" else "put")
        half = max(0.5, 0.02 * mid + 0.5)      # 教学价差：越虚值，价差占比越大
        rows.append({"行权价": K, "类型": kind, "买价": max(0.5, mid - half),
                     "卖价": mid + half, "中间价": mid,
                     "持仓量": int(1500 * np.exp(-abs(K - F) / 180)),
                     "成交量": int(260 * np.exp(-abs(K - F) / 140)) + (40 if K == F else 0)})
chain = pd.DataFrame(rows)
chain["价差占比"] = (chain["卖价"] - chain["买价"]) / chain["中间价"]
atm = float(strikes[np.argmin(np.abs(strikes - F))])
view = chain[(chain["行权价"] >= atm - 100) & (chain["行权价"] <= atm + 100)].copy()


def implied(price, K, kind):
    f = lambda v: black76(F, K, T, v, r, kind) - price
    try:
        return brentq(f, 1e-4, 2.0)
    except ValueError:
        return float("nan")


view["IV"] = [implied(m, k, "call" if t == "C" else "put")
              for m, k, t in zip(view["中间价"], view["行权价"], view["类型"])]
liquid = view[(view["价差占比"] < 0.06) & (view["成交量"] >= 10) & (view["持仓量"] >= 100)]
print(f"平值行权价 {atm:.0f}；以下为教学示例链，非真实报价")
print(liquid[["行权价", "类型", "买价", "卖价", "价差占比", "持仓量", "成交量", "IV"]]
      .to_string(index=False, formatters={"买价": "{:.1f}".format, "卖价": "{:.1f}".format,
                                          "价差占比": "{:.2%}".format, "IV": "{:.1%}".format}))
calls = liquid[liquid["类型"] == "C"].sort_values("行权价")
print(f"看涨 IV：低行权价 {calls.iloc[0]['IV']:.1%}，高行权价 {calls.iloc[-1]['IV']:.1%}"
      " → 低行权价更高，是 put 偏斜")


平值行权价 3500；以下为教学示例链，非真实报价
 行权价 类型    买价    卖价  价差占比  持仓量  成交量    IV
3400  C 140.1 146.8 4.70%  860  127 21.4%
3450  C 106.7 112.1 4.91% 1136  181 20.7%
3450  P  57.8  61.2 5.68% 1136  181 20.7%
3500  C  77.8  82.0 5.25% 1500  300 20.0%
3500  P  77.8  82.0 5.25% 1500  300 20.0%
3550  C  54.0  57.2 5.80% 1136  181 19.4%
3550  P 102.9 108.1 4.95% 1136  181 19.4%
3600  P 133.2 139.6 4.73%  860  127 18.8%
看涨 IV：低行权价 21.4%，高行权价 19.4% → 低行权价更高，是 put 偏斜



**读数与应用**：筛掉的正是两头虚值档——价差占比超标。所以“绝对值便宜”的虚值期权，实际成本可能更贵。IV 一列低行权价更高：市场给下跌保护更贵的价格，这就是商品期权常见的偏斜。

**常见错误**：

1. 只看权利金绝对值排序，不看价差占比。
2. 用买价卖价的中间价算收益，成交时却按不利方向。
3. 用一条链的 IV 直接比较不同到期日，忽略期限结构。
4. 拿很久没成交的报价当“市场价”。

**停下来想一想**：为什么虚值期权的价差占比通常更大？

<details><summary>对照你的思路</summary>

虚值期权权利金小，而最小变动价位和做市商的固定成本不随权利金同比缩小，所以价差相对于中间价的比例更高。同时虚值档的挂单更薄、对冲更难，做市商要更高的补偿。筛选链时要看“占比”，不能只看绝对价格。

</details>

---

## 6. 行权、指派与商品季节性

### 6.1 到期前：行权还是平仓

- **行权收益**：立即按 $K$ 换到期货头寸，毛收益等于内在价值；
- **平仓收益**：直接卖出期权，收益等于市场价。

若期权市场价**高于**内在价值（还有时间价值），平仓通常更优；若市场价**低于**内在价值（深实值期权因折现可能出现），行权可能更优。还要扣手续费、考虑行权后的期货保证金与资金占用。**不能只看“实值就行权”。**

### 6.2 行权之后发生什么

- 看涨行权：得到多头期货；看跌行权：得到空头期货；
- 期货头寸立刻进入保证金体系，按当日结算价盯市；
- 卖方被指派是**随机**的，不是按开仓顺序；
- 临近到期，实值期权可能自动行权、虚值作废（以交易所规则为准）；
- 行权手续费、行权后的资金占用都要算进“一手账本”。

### 6.3 商品季节性

农产品看种植/生长/收割与库存，能源看取暖季与驾驶季，金属看开工与库存周期。检验方法：按月份分组算均值、看多年稳定性、做 $t$ 检验、留样本外。

季节性的三个陷阱：**样本短**（10 年只有 10 个观测）、**多重检验**（12 个月里挑最显著的）、**结构变化**（产能、政策、贸易流改变后，老季节性消失）。看到“2 月必涨”先问一句：经济逻辑是什么？样本外还成立吗？

### 6.4 代码：行权判断、行权后保证金与季节性



In [6]:

import numpy as np

prem_paid, K, mult = 80.0, 3600.0, 10
print("买入一手行权价 3600 的看涨，权利金成本 80 元/吨")
print(f"{'标的价':>7}{'内在价值':>9}{'期权市场价':>10}{'行权净利':>9}{'平仓净利':>9}{'更优':>5}")
for FT, mkt in [(3500, 20), (3650, 95), (4000, 398)]:
    intrinsic = max(FT - K, 0)
    exercise = (intrinsic - prem_paid) * mult
    close = (mkt - prem_paid) * mult
    print(f"{FT:>7.0f}{intrinsic:>9.0f}{mkt:>10.0f}{exercise:>9.0f}{close:>9.0f}"
          f"{'平仓' if close >= exercise else '行权':>5}")

print(f"若在 4000 行权，得到多头期货，保证金约 {4000 * mult * 0.10:.0f} 元（教学示例 10%）")

rng = np.random.default_rng(4201)
effect = np.array([0.010, -0.015, 0.005, 0.012, 0.000, -0.010,
                   -0.020, 0.008, 0.015, 0.005, -0.005, 0.018])
rets = effect[None, :] + rng.normal(0, 0.03, (10, 12))
mean = rets.mean(axis=0)
tstat = mean / (rets.std(axis=0, ddof=1) / np.sqrt(10))
print("教学季节性数据（10 年 × 12 月，合成数据，非真实行情）：")
print(f"{'月份':>4}{'平均收益':>9}{'t值':>7}")
for m in range(12):
    print(f"{m + 1:>4}{mean[m]:>8.2%}{tstat[m]:>7.2f}")


买入一手行权价 3600 的看涨，权利金成本 80 元/吨
    标的价     内在价值     期权市场价     行权净利     平仓净利   更优
   3500        0        20     -800     -600   平仓
   3650       50        95     -300      150   平仓
   4000      400       398     3200     3180   行权
若在 4000 行权，得到多头期货，保证金约 4000 元（教学示例 10%）
教学季节性数据（10 年 × 12 月，合成数据，非真实行情）：
  月份     平均收益     t值
   1  -0.39%  -0.49
   2  -1.60%  -2.57
   3  -0.12%  -0.13
   4   1.54%   1.47
   5   0.62%   0.68
   6  -1.15%  -1.58
   7  -1.56%  -1.77
   8   1.91%   1.97
   9   0.79%   0.90
  10   1.48%   1.37
  11  -0.23%  -0.37
  12   2.77%   3.04



**读数与应用**：三个情形说明“实值不等于该行权”：3500 时只有时间价值，行权净亏 800 元；3650 时市场价高于内在价值，平仓更优；4000 时市场价低于内在价值，行权多赚 20 元——这是深实值期权的折现效应。季节性表里 2 月（−2.57）和 12 月（+3.04）的 $|t|$ 超过 2，但 10 年只有 10 个观测，**显著不等于可交易**。

**常见错误**：

1. 认为“实值就应该行权”，忽略时间价值与折现。
2. 行权后忘了期货保证金，以为行权就落袋为安。
3. 用 12 个月里最显著的月份做策略，不做多重检验。
4. 拿合成数据或短样本的季节性直接上实盘。

---

## 本章小结

### 一张表

| 概念 | 做法 | 用途 |
|---|---|---|
| 合约要素 | 标的、乘数、最小变动、行权方式、最后交易日 | 算账前提 |
| 一手名义 | 期货价 × 乘数 | 资金与容量 |
| 一跳价值 | 最小变动 × 乘数 | 报价精度与成本 |
| 卖方保证金 | max(权利金+期货保证金−0.5×虚值额, 权利金+0.5×期货保证金) | 卖方资金占用 |
| 一手盈亏 | (支付 − 权利金) × 乘数 | 买方账本 |
| 期权链 | 行权价 + 买卖价 + 持仓/成交 + IV | 找可交易档位 |
| 行权 | 内在价值 vs 市场价 | 到期管理 |
| 季节性 | 月度均值、稳定性、样本外 | 辅助假设 |

### 八句话

1. 商品期权的标的是期货，行权得到的是期货头寸。
2. 一手账本第一步永远是乘乘数。
3. 行权价间距决定链的密度，最小变动决定报价精度。
4. 最后交易日在标的交割月之前，别拿它当流动性窗口。
5. 卖方保证金是“期货保证金 + 权利金 − 虚值额”，不是固定比例。
6. 虚值期权的价差占比更大，筛选要看比例不看绝对值。
7. 实值不等于该行权，要看时间价值与折现。
8. 季节性容易过拟合，样本外和逻辑缺一不可。

---

## 本章习题

**1.** 某合约：报价单位元/吨、合约乘数 10、最小变动 1.0 元/吨。一跳价值是多少？行权方式对策略有什么影响？

**2.** 权利金报价 45 元/吨，乘数 10，买 3 手。一手权利金多少？到期期货价 3700、行权价 3500 时，一手净利多少（忽略费用）？

**3.** 教学示例：$F=3500$、$K=3600$ 的看涨，权利金 55 元/吨，乘数 10，期货保证金率 10%。求卖方一手保证金。

**4.** 教学示例：$F=3500$、$K=3400$ 的看跌，权利金 45 元/吨，乘数 10，期货保证金率 10%。求卖方一手保证金。

**5.** 买入看涨，行权价 3500、报价 45 元/吨；买入看跌，行权价 3500、报价 30 元/吨。各自的盈亏平衡点（每吨）是多少？

**6.** 到期前：$F=4000$、$K=3600$ 的看涨，内在价值 400 元/吨，期权市场价 398 元/吨，权利金成本 80 元/吨，乘数 10。行权与平仓哪个更优？差多少（忽略费用）？

**7.** 一条期权链上，为什么虚值期权的买卖价差占比通常更大？筛选时应怎么处理？

**8.** 某品种过去 10 年的月度收益里，7 月平均 −2%、$t$ 值 −2.3。能不能直接做“7 月做空”？说明理由。

<details><summary>参考答案</summary>

**1.** 一跳价值 $=1.0\times10=10$ 元。行权方式决定能否提前行权：美式要考虑提前行权与被指派，欧式只需管到期日。

**2.** 一手权利金 $=45\times10=450$ 元，3 手 1350 元；到期支付 $(3700-3500)\times10=2000$ 元，一手净利 $2000-450=1550$ 元。

**3.** 权利金市值 550，期货保证金 3500，虚值额 $\max(3600-3500,0)\times10=1000$；保证金 $=\max(550+3500-500,\ 550+1750)=3550$ 元。

**4.** 权利金市值 450，期货保证金 3500，虚值额 $\max(3500-3400,0)\times10=1000$；保证金 $=\max(450+3500-500,\ 450+1750)=3450$ 元。

**5.** 看涨平衡点 $3500+45=3545$ 元/吨；看跌平衡点 $3500-30=3470$ 元/吨。

**6.** 行权净利 $=(400-80)\times10=3200$ 元；平仓净利 $=(398-80)\times10=3180$ 元；行权更优，多 20 元（还要扣行权手续费、考虑资金占用与期货保证金）。

**7.** 虚值期权权利金小，而最小变动与做市成本不随权利金同比缩小；挂单更薄、对冲更难。筛选时看价差占中间价的比例，并叠加成交量/持仓量过滤。

**8.** 不能。10 年只有 10 个观测，$t$ 值 −2.3 又是在 12 个月里挑出来的，有多重检验问题；还要看分年稳定性、经济逻辑和样本外。显著只是“值得研究”，不是“可以下单”。

</details>

---

## 参考资料

- 上海期货交易所、大连商品交易所、郑州商品交易所、上海国际能源交易中心、广州期货交易所官网（合约细则与保证金公告）。
- Hull, J. C. *Options, Futures, and Other Derivatives*（期货期权定价与保证金）。
- 中国期货业协会《期货及衍生品基础》《期货及衍生品分析与应用》。
- 第09、24、26、27、28章。
